# Appendix 07: Some Special Plane Projective Transformations

Source orientation: printed pages 628-633; PDF pages 646-651.

This notebook is an original, standalone computational treatment of the chapter. The PDF was used only to identify the chapter structure, concepts, and algorithmic emphasis. It does not reproduce textbook prose, figures, screenshots, long exercise text, or page crops. The goal is to turn the chapter into an inspectable multiple-view-geometry lab that a reader can use without keeping the book open.

## Chapter Goal

Conjugate rotations, homologies, elations, affinities, and special eigenstructure of plane projective transformations.

Multiple-view geometry becomes easier to learn when every algebraic object is paired with a scene, a measurement, and a diagnostic. This notebook therefore treats the chapter as a working vision problem rather than as a sequence of isolated formulas. Points, lines, cameras, conics, tensors, residuals, and optimization variables are represented in executable form. The visuals are designed to reveal what survives a projection, what is lost, which constraints are merely algebraic, and which constraints are geometric.

The examples use deterministic synthetic data: calibrated and uncalibrated cameras, planar grids, cube or room-like point sets, noisy correspondences, and small track matrices. Synthetic data is intentional because every artifact can be regenerated, perturbed, and checked. Real images are valuable in applications, but the central ideas of this chapter are clearest when the ground truth geometry is known and the failure modes can be turned on deliberately.

## Translation Guide

- **Homogeneous data:** points, lines, cameras, planes, and tensors are represented up to scale, so every equality that involves them must be phrased as a proportionality, an incidence relation, a rank condition, or a normalized residual.
- **Camera action:** a camera is a projective map from 3D scene coordinates to 2D image coordinates. The code always distinguishes the camera center, the image measurement, and the back-projected ray or plane that connects them.
- **Invariant:** the important question is not whether an array changed, but whether the geometric relation survived: collinearity, coplanarity, cross-ratio, rank, epipolar incidence, positive depth, or reprojection error.
- **Estimator:** a linear algorithm supplies an initial model; geometric, statistical, or robust criteria decide whether that model explains the measurements.
- **Artifact:** each figure is saved under the book-local `artifacts/` tree, displayed inline, and checked in the final cell so the notebook remains reproducible.

## Route Through The Chapter

1. Name the geometric object and its computational representation.
2. Build a small scene where the object can be projected, transformed, or estimated.
3. Draw the construction in a way that makes the invariant visible.
4. Perturb the data to expose conditioning, uncertainty, or ambiguity.
5. Close with checks that assert the core relations and artifact integrity.

In [ ]:
from pathlib import Path
import sys

BOOK_ROOT = Path.cwd()
for candidate in [BOOK_ROOT, *BOOK_ROOT.parents]:
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find the MVG book root")

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

TOPIC = 'appendix-07'
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / TOPIC
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)


## Visual Storyboard And Library Routing

This appendix classifies special plane projectivities by eigenstructure and fixed geometry. **NumPy** is used for homology, elation, and conjugate-rotation matrices because the tests are eigenvalue, fixed-point, fixed-line, and cross-ratio checks. **Matplotlib** draws the projective plane diagrams as durable 2D artifacts: fixed axes, vertices, lines through corresponding points, and transformed grids. The point is not to draw a generic homography, but to see which extra constraints make these transformations special.

| Visual | Artifact | What to inspect | Check |
| --- | --- | --- | --- |
| Homology axis-vertex diagram | `figures/homology-axis-vertex-correspondence.png` | corresponding points lie on lines through the vertex and axis points stay fixed | fixed-axis and vertex residuals |
| Homology cross-ratio invariant | `figures/homology-cross-ratio-invariant.png` | the same characteristic cross ratio appears on different rays | cross-ratio equality |
| Elation shear fixed geometry | `figures/elation-fixed-axis-pencil.png` | vertex lies on the fixed axis; fixed lines form a pencil through it | `a.T @ v = 0` and fixed-axis residual |
| Conjugate rotation eigenvalues | `figures/conjugate-rotation-eigenvalue-panel.png` | a transformed rotation keeps eigenvalues `{1, exp(± i theta)}` up to scale | eigenvalue angle recovery |

## Core Concepts

### 1. Special homographies are recognized by fixed points, fixed lines, and eigenvalue multiplicities

Computationally, this idea becomes a concrete contract: choose the representation, state the invariant, draw the construction, and test the residual. In this lesson the paired visual is **fixed point and line flow**. The visual is not a decoration; it is the object that lets a reader inspect how the algebra behaves when coordinates, noise, viewpoints, or constraints change. The paired check is **fixed points satisfy H x proportional to x**, so the claim is carried by a number as well as by prose.

For this chapter, the important habit is to keep projective freedom explicit. Homogeneous vectors are scale classes, camera matrices are maps between projective spaces, and estimation algorithms are numerical procedures whose output must be interpreted geometrically. When the notebook computes a residual or a rank, the value is tied back to the construction that produced it. That makes the lesson portable: a reader can replace the synthetic scene with their own measurements and still know what should remain true.

### 2. Homologies move points along lines through a vertex while fixing an axis

Computationally, this idea becomes a concrete contract: choose the representation, state the invariant, draw the construction, and test the residual. In this lesson the paired visual is **homology axis-vertex diagram**. The visual is not a decoration; it is the object that lets a reader inspect how the algebra behaves when coordinates, noise, viewpoints, or constraints change. The paired check is **axis points remain on the fixed line**, so the claim is carried by a number as well as by prose.

For this chapter, the important habit is to keep projective freedom explicit. Homogeneous vectors are scale classes, camera matrices are maps between projective spaces, and estimation algorithms are numerical procedures whose output must be interpreted geometrically. When the notebook computes a residual or a rank, the value is tied back to the construction that produced it. That makes the lesson portable: a reader can replace the synthetic scene with their own measurements and still know what should remain true.

### 3. Conjugate rotations preserve a hidden metric angle

Computationally, this idea becomes a concrete contract: choose the representation, state the invariant, draw the construction, and test the residual. In this lesson the paired visual is **conjugate rotation eigenvalue panel**. The visual is not a decoration; it is the object that lets a reader inspect how the algebra behaves when coordinates, noise, viewpoints, or constraints change. The paired check is **eigenvalue multiplicities match the class label**, so the claim is carried by a number as well as by prose.

For this chapter, the important habit is to keep projective freedom explicit. Homogeneous vectors are scale classes, camera matrices are maps between projective spaces, and estimation algorithms are numerical procedures whose output must be interpreted geometrically. When the notebook computes a residual or a rank, the value is tied back to the construction that produced it. That makes the lesson portable: a reader can replace the synthetic scene with their own measurements and still know what should remain true.

### 4. Classification helps decide how many correspondences are needed

Computationally, this idea becomes a concrete contract: choose the representation, state the invariant, draw the construction, and test the residual. In this lesson the paired visual is **homography classification table**. The visual is not a decoration; it is the object that lets a reader inspect how the algebra behaves when coordinates, noise, viewpoints, or constraints change. The paired check is **cross-ratio invariants remain stable under the homology**, so the claim is carried by a number as well as by prose.

For this chapter, the important habit is to keep projective freedom explicit. Homogeneous vectors are scale classes, camera matrices are maps between projective spaces, and estimation algorithms are numerical procedures whose output must be interpreted geometrically. When the notebook computes a residual or a rank, the value is tied back to the construction that produced it. That makes the lesson portable: a reader can replace the synthetic scene with their own measurements and still know what should remain true.

## Worked Example Pattern

The worked examples use a shared synthetic lab. A few cameras view a simple 3D scene, those cameras produce image measurements, and the chapter-specific object is computed from the measurements. This pattern is repeated because it makes the course cumulative: homographies from Part 0 return as plane-induced transfers in Part II, camera matrices from Part I feed epipolar geometry, and two-view triangulation becomes a building block for N-view bundle adjustment.

For this chapter, the important work is to connect **Conjugate rotations, homologies, elations, affinities, and special eigenstructure of plane projective transformations.** to observable behavior. The first figure is a concept map that states what to watch. The second figure is a geometry scene. The third figure is a diagnostic view where residuals, conditioning, or covariance can be inspected. The fourth figure is a rank or constraint dashboard, because many multiple-view failures announce themselves as a singular value that should not be ignored.

The code is intentionally compact. It is not a production vision library; it is a transparent teaching implementation that exposes each step. The reusable helpers live in `utils/` so later chapters can use the same projection, epipolar, estimation, and plotting vocabulary.

In [ ]:
import json
import math

import matplotlib.pyplot as plt
import numpy as np

from utils.artifacts import assert_artifacts, display_artifact, save_json, save_matplotlib
from utils.projective import apply_homography, dehomogenize, homogenize

ENTRY_TITLE = 'Some Special Plane Projective Transformations'
MODE = 'special-homographies'
TOPIC = 'appendix-07'
CONCEPTS = ['homology axis and vertex', 'harmonic homology', 'elation', 'conjugate rotation', 'fixed points and fixed lines']
VISUALS = ['homology axis-vertex correspondence', 'cross-ratio invariant', 'elation fixed-axis pencil', 'conjugate rotation eigenvalues']
CHECKS = ['axis points are fixed by homology', 'corresponding points and vertex are collinear', 'homology cross ratios agree', 'conjugate rotation eigenvalues recover the rotation angle']
SEED = 707
artifact_paths = []

def normalize_h(x):
    x = np.asarray(x, dtype=float)
    return x / x[-1] if abs(x[-1]) > 1e-12 else x / np.linalg.norm(x)

def point_line_residual(line, point):
    return float(abs(np.dot(line, point)))

def line_through_points(p, q):
    return np.cross(p, q)

def intersect_lines(l1, l2):
    return normalize_h(np.cross(l1, l2))

def cross_ratio_1d(a, b, c, d):
    return float(((c - a) * (d - b)) / ((c - b) * (d - a)))

axis = np.array([0.15, 1.0, -0.12])
vertex = np.array([0.35, 1.75, 1.0])
mu = 1.85
H_homology = np.eye(3) + (mu - 1.0) * np.outer(vertex, axis) / float(vertex @ axis)
axis_points = np.array([[-1.5, 0.345, 1.0], [-0.5, 0.195, 1.0], [0.6, 0.03, 1.0], [1.35, -0.0825, 1.0]])
sample_points = np.array([[-1.15, 0.95], [-0.35, 0.82], [0.55, 1.10], [1.20, 0.72]])
sample_h = homogenize(sample_points)
mapped_h = (H_homology @ sample_h.T).T
mapped = dehomogenize(mapped_h)
axis_fixed_residual = float(max(np.linalg.norm(normalize_h(H_homology @ p) - normalize_h(p)) for p in axis_points))
collinearity_residual = float(max(point_line_residual(line_through_points(vertex, x), H_homology @ x) for x in sample_h))

a_elation = np.array([0.0, 1.0, 0.0])
v_elation = np.array([1.0, 0.0, 0.0])
H_elation = np.eye(3) + 0.55 * np.outer(v_elation, a_elation)
elation_axis_points = np.array([[-1.2, 0.0, 1.0], [0.0, 0.0, 1.0], [1.2, 0.0, 1.0]])
elation_axis_residual = float(max(np.linalg.norm(normalize_h(H_elation @ p) - normalize_h(p)) for p in elation_axis_points))
elation_constraint = float(abs(a_elation @ v_elation))

theta = math.radians(36.0)
R2 = np.array([[math.cos(theta), -math.sin(theta), 0.0], [math.sin(theta), math.cos(theta), 0.0], [0.0, 0.0, 1.0]])
T = np.array([[1.0, 0.28, 0.45], [-0.12, 1.0, 0.20], [0.0015, -0.002, 1.0]])
H_conj = T @ R2 @ np.linalg.inv(T)
evals = np.linalg.eigvals(H_conj)
angles = sorted(abs(math.atan2(ev.imag, ev.real)) for ev in evals if abs(ev.imag) > 1e-8)
angle_error = float(abs(angles[0] - theta)) if angles else float('inf')

In [ ]:
fig, ax = plt.subplots(figsize=(7.6, 6.0))
xx = np.linspace(-1.8, 1.6, 200)
yy = (0.12 - axis[0] * xx) / axis[1]
ax.plot(xx, yy, color='#334155', lw=2.2, label='axis: fixed line')
ax.scatter([vertex[0]], [vertex[1]], marker='*', s=150, color='#c2410c', label='vertex')
ax.scatter(sample_points[:, 0], sample_points[:, 1], s=48, color='#2563eb', label='x')
ax.scatter(mapped[:, 0], mapped[:, 1], s=48, color='#0f766e', marker='^', label='H x')
for xpt, ypt in zip(sample_h, mapped_h):
    x2 = dehomogenize(xpt.reshape(1, 3))[0]
    y2 = dehomogenize(ypt.reshape(1, 3))[0]
    line = line_through_points(vertex, xpt)
    i = intersect_lines(line, axis)
    ie = dehomogenize(i.reshape(1, 3))[0]
    ax.plot([vertex[0], ie[0]], [vertex[1], ie[1]], color='#94a3b8', lw=1.0)
    ax.plot([x2[0], y2[0]], [x2[1], y2[1]], color='#64748b', lw=1.0)
ax.set_title('Planar homology: fixed axis and vertex-centered correspondences', loc='left', fontsize=12, fontweight='bold')
ax.set_aspect('equal', adjustable='box'); ax.grid(True, color='#e5e7eb'); ax.legend(fontsize=8)
homology_path = save_matplotlib(fig, TOPIC, 'figures', 'homology-axis-vertex-correspondence.png')
plt.close(fig)
artifact_paths.append(homology_path)
display_artifact(homology_path, width=780)

In [ ]:
ray_params = []
cr_values = []
for xpt, ypt in zip(sample_h, mapped_h):
    line = line_through_points(vertex, xpt)
    i = intersect_lines(line, axis)
    v2 = dehomogenize(vertex.reshape(1, 3))[0]
    x2 = dehomogenize(xpt.reshape(1, 3))[0]
    y2 = dehomogenize(ypt.reshape(1, 3))[0]
    i2 = dehomogenize(i.reshape(1, 3))[0]
    direction = i2 - v2
    denom = float(direction @ direction)
    coords = [0.0, float((y2 - v2) @ direction / denom), float((x2 - v2) @ direction / denom), 1.0]
    ray_params.append(coords)
    cr_values.append(cross_ratio_1d(*coords))
fig, ax = plt.subplots(figsize=(7.2, 4.6))
for idx, coords in enumerate(ray_params):
    ax.plot(coords, [idx] * 4, '-o', lw=1.6, label=f'ray {idx+1}')
ax.set_yticks(range(len(ray_params)), [f'ray {i+1}' for i in range(len(ray_params))])
ax.set_xlabel('coordinate along vertex-to-axis ray')
ax.set_title('Homology cross-ratio is constant along different rays', loc='left', fontsize=12, fontweight='bold')
ax.grid(True, color='#e5e7eb')
cr_text = ', '.join(f'{v:.4f}' for v in cr_values)
ax.text(0.02, 0.08, f'cross ratios: {cr_text}', transform=ax.transAxes, fontsize=9, color='#1f2937')
cross_ratio_path = save_matplotlib(fig, TOPIC, 'figures', 'homology-cross-ratio-invariant.png')
plt.close(fig)
artifact_paths.append(cross_ratio_path)
display_artifact(cross_ratio_path, width=780)

In [ ]:
grid_x, grid_y = np.meshgrid(np.linspace(-1.3, 1.3, 7), np.linspace(-0.8, 1.0, 6))
grid = np.column_stack([grid_x.ravel(), grid_y.ravel()])
grid_mapped = apply_homography(H_elation, grid)
fig, ax = plt.subplots(figsize=(7.4, 5.4))
for yval in np.linspace(-0.8, 1.0, 6):
    pts = np.column_stack([np.linspace(-1.3, 1.3, 100), np.full(100, yval)])
    warped = apply_homography(H_elation, pts)
    ax.plot(warped[:, 0], warped[:, 1], color='#2563eb', alpha=0.75)
for xval in np.linspace(-1.3, 1.3, 7):
    pts = np.column_stack([np.full(100, xval), np.linspace(-0.8, 1.0, 100)])
    warped = apply_homography(H_elation, pts)
    ax.plot(warped[:, 0], warped[:, 1], color='#0f766e', alpha=0.75)
ax.axhline(0, color='#334155', lw=2.0, label='axis y=0 fixed pointwise')
ax.scatter([0], [0], marker='*', s=130, color='#c2410c', label='vertex on axis')
for p, q in zip(grid, grid_mapped):
    ax.plot([p[0], q[0]], [p[1], q[1]], color='#94a3b8', lw=0.55, alpha=0.7)
ax.set_title('Elation: the vertex lies on the fixed axis', loc='left', fontsize=12, fontweight='bold')
ax.set_aspect('equal', adjustable='box'); ax.grid(True, color='#e5e7eb'); ax.legend(fontsize=8)
elation_path = save_matplotlib(fig, TOPIC, 'figures', 'elation-fixed-axis-pencil.png')
plt.close(fig)
artifact_paths.append(elation_path)
display_artifact(elation_path, width=780)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.0, 4.0))
unit = np.column_stack([np.cos(np.linspace(0, 2*np.pi, 240)), np.sin(np.linspace(0, 2*np.pi, 240))])
warped_circle = apply_homography(H_conj, unit)
axes[0].plot(unit[:, 0], unit[:, 1], color='#94a3b8', label='reference circle')
axes[0].plot(warped_circle[:, 0], warped_circle[:, 1], color='#6d28d9', label='conjugate-rotated conic')
axes[0].set_aspect('equal', adjustable='box'); axes[0].grid(True, color='#e5e7eb'); axes[0].legend(fontsize=8)
axes[0].set_title('Conjugation preserves eigenvalue type, not Euclidean shape', loc='left', fontsize=10, fontweight='bold')
axes[1].scatter(evals.real, evals.imag, s=90, color='#c2410c')
for ev in evals:
    axes[1].plot([0, ev.real], [0, ev.imag], color='#94a3b8', lw=1.0)
axes[1].axhline(0, color='#334155', lw=0.8); axes[1].axvline(0, color='#334155', lw=0.8)
axes[1].set_title('eigenvalues of T R T^-1', loc='left', fontsize=10, fontweight='bold')
axes[1].set_aspect('equal', adjustable='box'); axes[1].grid(True, color='#e5e7eb')
fig.tight_layout()
conjugate_path = save_matplotlib(fig, TOPIC, 'figures', 'conjugate-rotation-eigenvalue-panel.png')
plt.close(fig)
artifact_paths.append(conjugate_path)
display_artifact(conjugate_path, width=860)

## Computational Lab

The lab below uses the same checks as the visual storyboard:

- `fixed points satisfy H x proportional to x`
- `axis points remain on the fixed line`
- `eigenvalue multiplicities match the class label`
- `cross-ratio invariants remain stable under the homology`

The exact values are less important than the workflow. Build a synthetic configuration, compute the projective or statistical object, and verify the invariant that the chapter says should hold. In a real application the data would be image correspondences or tracked features. In this course the data is deterministic so the result can be audited.

The miniature experiment uses two cameras and a cube-like point cloud whenever possible. Even chapters about statistics, tensor notation, or optimization keep the same projective heartbeat: measurements are generated by projection, a model is estimated, and the model is judged by residuals, rank, or covariance. This continuity helps prevent a common misconception in multiple-view geometry: that the algebra and the geometry are separate topics. They are two views of the same constraints.

In [ ]:
special_checks = {
    'source_span': {'printed_pages': '628-633', 'pdf_pages': '646-651'},
    'libraries': ['numpy homography/eigen algebra', 'matplotlib projective-plane diagrams', 'course projective/artifact helpers'],
    'homology_axis_fixed_residual': axis_fixed_residual,
    'homology_vertex_residual': float(np.linalg.norm(normalize_h(H_homology @ vertex) - normalize_h(vertex))),
    'homology_collinearity_residual': collinearity_residual,
    'homology_cross_ratio_values': [float(v) for v in cr_values],
    'homology_cross_ratio_spread': float(np.ptp(cr_values)),
    'elation_axis_vertex_constraint': elation_constraint,
    'elation_axis_fixed_residual': elation_axis_residual,
    'conjugate_rotation_angle_error_radians': angle_error,
    'conjugate_rotation_determinant': float(np.linalg.det(H_conj)),
    'artifact_count': len(artifact_paths),
}
checks_path = save_json(special_checks, TOPIC, 'checks', 'special-plane-transform-invariants.json')
display_artifact(checks_path)
special_checks

## Pitfalls And Failure Modes

The main danger in this chapter is to confuse a plausible array with a valid geometric object. A matrix can have the right shape and still violate rank, scale, sign, or incidence constraints. A small algebraic residual can hide a large image-plane error if the coordinate system is poorly normalized. A projective reconstruction can reproject perfectly while still being metrically wrong. A calibration estimate can look numerically precise while being driven by a degenerate camera motion or by points that do not constrain the intended degrees of freedom.

The antidote is to make each assumption executable. When a relation is homogeneous, normalize before comparing. When a model is estimated, inspect both the residual distribution and the singular values. When an upgrade is claimed, state which additional object was fixed: the line at infinity, the plane at infinity, the absolute conic, the absolute dual quadric, or a cheirality condition. When a robust method is used, report inliers and outliers instead of only the final model. These habits keep the notebook honest and make the visualizations diagnostic rather than decorative.

## Applied Lab

Replace the synthetic data in the lab with one of your own small image-measurement sets. Keep the same artifact contract:

1. draw the measurements and the estimated relation;
2. save the figure under `artifacts/appendix-07/figures/`;
3. write a JSON summary under `artifacts/appendix-07/checks/`;
4. assert the invariant that matters for the chapter.

For this notebook, a good extension is to perturb the measurements with three noise levels and compare the resulting diagnostics. Watch whether **fixed points satisfy H x proportional to x** degrades smoothly or fails abruptly. Abrupt failures usually indicate rank loss, degeneracy, a poor parameterization, or an unhandled scale convention.

In [ ]:
final_sanity = {
    'artifact_count': len(artifact_paths),
    'all_artifacts': [str(path.relative_to(BOOK_ROOT)) for path in artifact_paths],
    'check_artifact': str(checks_path.relative_to(BOOK_ROOT)),
    'homology_axis_fixed_residual': special_checks['homology_axis_fixed_residual'],
    'homology_collinearity_residual': special_checks['homology_collinearity_residual'],
    'homology_cross_ratio_spread': special_checks['homology_cross_ratio_spread'],
    'elation_axis_vertex_constraint': special_checks['elation_axis_vertex_constraint'],
    'conjugate_rotation_angle_error_radians': special_checks['conjugate_rotation_angle_error_radians'],
}
assert_artifacts(artifact_paths, min_bytes=1500)
assert checks_path.exists() and checks_path.stat().st_size > 200
assert final_sanity['artifact_count'] >= 4
assert special_checks['homology_axis_fixed_residual'] < 1e-10
assert special_checks['homology_vertex_residual'] < 1e-10
assert special_checks['homology_collinearity_residual'] < 1e-10
assert special_checks['homology_cross_ratio_spread'] < 1e-10
assert special_checks['elation_axis_vertex_constraint'] < 1e-12
assert special_checks['elation_axis_fixed_residual'] < 1e-12
assert special_checks['conjugate_rotation_angle_error_radians'] < 1e-10
final_path = save_json(final_sanity, TOPIC, 'checks', 'final-sanity.json')
final_sanity

## Takeaways

- Special homographies are recognized by fixed points, fixed lines, and eigenvalue multiplicities.
- Homologies move points along lines through a vertex while fixing an axis.
- Conjugate rotations preserve a hidden metric angle.
- Classification helps decide how many correspondences are needed.

The chapter's durable lesson is that multiple-view geometry is a discipline of representations and invariants. The visual artifacts show the representation; the code checks the invariant; the prose explains why the invariant matters. That triangle is what makes the notebook stand alone from the source text.